<a href="https://colab.research.google.com/github/ollihansen90/Mathe-SH/blob/main/2D_Zellautomaten.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2D-Zellautomaten - Conway's Game of Life
- Kurzbeschreibung
    - keine Regeln
    - kurzer Recap zu 1D-Zellautomat Rule 30
    - Unterschied hier: Anzahl Nachbarn, nicht die Konstellation (das macht es zu einem *Subset*)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Die Welt
- 2D-Numpy-Array (mit `zeros`)
- einige Zellen leben (1), andere sind tot (0)

### Aufgabe
- Weltgröße 5x5
- Strich GENAU in der Mitte von oben nach unten

In [ ]:
N = 5
welt = np.zeros([N,N])

welt[1,2] = 1
welt[2,2] = 1
welt[3,2] = 1

plt.figure()
plt.imshow(welt)
plt.show()

## Padding
- hier mit `np.pad`
    - Kurzbeschreibung der Parameter (https://numpy.org/devdocs/reference/generated/numpy.pad.html)
    - nähere Beschreibung des Parameters `mode` (constant, mean, wrap,...)
    - KEINE eigene Implementierung (zu kompliziert)

In [ ]:
welt_pad = np.pad(welt, 2, mode="wrap")

plt.figure()
plt.imshow(welt_pad)
plt.show()

## Welt-Update
- nur ein Schritt
- als vierfache For-Schleife, sehr ineffizient
- Regeln sollen implementiert werden

In [ ]:
welt_pad = np.pad(welt, 1, mode="wrap")
welt_neu = np.zeros([N,N])

for i in range(1,N+1):
    for j in range(1,N+1):
        summe = 0
        for ii in range(-1,2):
            for jj in range(-1,2):
                if ii==jj and ii==0:
                    continue
                summe += welt_pad[i+ii, j+jj]

        wert = 0
        if welt[i-1, j-1] == 1 and (summe==2 or summe==3):
            wert = 1
        if welt[i-1, j-1] == 0 and summe==3:
            wert = 1
        welt_neu[i-1,j-1] = wert

plt.figure()
plt.subplot(1,2,1)
plt.imshow(welt)
plt.subplot(1,2,2)
plt.imshow(welt_neu)
plt.show()

## Update über Parallelisierung
- Idee: jedes Feld wird neun Mal besucht
- Parallelisierung durch "Elternliste"
- Ergebnis: 9x5x5-Tensor mit den neun Eltern pro Pixel auf 5x5-Feld

### Aufgabe: Funktion `eltern`
    - Padding (!)
    - `stack`


In [ ]:
def eltern(welt):
    welt_pad = np.pad(welt, 1, mode="wrap")
    weltenliste = []
    for i in range(3):
        for j in range(3):
            weltenliste.append(welt_pad[i:len(welt)+i, j:len(welt)+j])
    return np.stack(weltenliste)

### Aufgabe: Updatefunktion
- erhält Elternwelt und Kernel (quasi die Gewichtung, genau wie bei Faltung, Kurzbeschreibung dazu)
- gewichtete Summe als Ausgabe


In [ ]:
def update(elternwelt, kernel):
    weltensumme = 0
    for k in range(len(kernel)):
        weltensumme += elternwelten[k]*kernel[k]

    welt_neu = np.zeros([N,N])
    welt_neu[weltensumme==8+2]  = 1
    welt_neu[weltensumme==8+3]  = 1
    welt_neu[weltensumme==3]  = 1
    return welt_neu

## Simulation
- Parameter: N=100 (also 100x100-Welt), Seed (0), Startwelt, Kernel

### Aufgabe: initialisiere "Glider" oben links in der Ecke
- am besten mit Bild

```
N = 100
welt = np.zeros([N,N])
welt[1,3] = 1
welt[2,3] = 1
welt[3,3] = 1
welt[3,2] = 1
welt[2,1] = 1
```

In [ ]:
np.random.seed(0)
N = 100
welt = np.zeros([N,N])
welt[1,3] = 1
welt[2,3] = 1
welt[3,3] = 1
welt[3,2] = 1
welt[2,1] = 1

welt = np.random.randn(N,N)
welt[welt>0.5] = 1
welt[welt<=0.5] = 0

kernel = np.ones(9)
kernel[4] = 8
video = []

for i in range(500):
    elternwelten = eltern(welt)
    welt_neu = update(elternwelten, kernel)

    video.append(welt_neu.copy()*255)
    welt = welt_neu

import imageio.v3 as iio
iio.imwrite(
    "animation.gif",
    video,
    duration=100,  # Anzeigedauer pro Bild in Millisekunden
    loop=0         # 0 = Endlosschleife
)

from IPython.display import Image, display

display(Image(filename="animation.gif",width=300, height=300))

## Verallgemeinerung auf $[0,1]$
- auf $[0,1]$ mit zufälligem Kernel
- ACHTUNG: Epilepsiewarnung!

In [ ]:
def eltern(welt):
    welt_pad = np.pad(welt, 1, mode="wrap")
    weltenliste = []
    for i in range(3):
        for j in range(3):
            weltenliste.append(welt_pad[i:len(welt)+i, j:len(welt)+j])
    return np.stack(weltenliste)

np.random.seed(10)
N = 100
welt = np.zeros([N,N])

welt[1,3] = 1
welt[2,3] = 1
welt[3,3] = 1
welt[3,2] = 1
welt[2,1] = 1

kernel = 2*np.random.randn(9)
welt = np.random.rand(N,N)

print(kernel.reshape(3,3))
video = []

for i in range(1000):
    weltensumme = 0

    elternwelten = eltern(welt)
    for k in range(len(kernel)):
        weltensumme += elternwelten[k]*kernel[k]

    welt_neu = 1/(1 + np.exp(-weltensumme))
    video.append(welt_neu.copy()*255)
    welt = welt_neu

import imageio.v3 as iio
iio.imwrite(
    "animation.gif",
    video,
    duration=100,  # Anzeigedauer pro Bild in Millisekunden
    loop=0         # 0 = Endlosschleife
)

from IPython.display import Image, display

display(Image(filename="animation.gif",width=300, height=300))